# 03 — E2: position bias, two protocols (W9)

**Free-tier version**: default judges are `gemini-3.8-flash` (Google AI Studio free tier -- the paper's `gemini-2.5-flash` is retired for new users as of 2026-11, Google's own API error pointed at this replacement) and `llama-3.3-70b-versatile` via **Groq** (free, no credit card). Llama 3.3-70B is already one of the paper's 5 judges, so that one only swaps the *endpoint* (Groq instead of Vertex AI); the Gemini swap is a real model-generation confound, not just an endpoint change -- both are recorded in `config.yaml`. Swap `claude-sonnet-4-5` back into `config.yaml`'s judge list later if you add Anthropic billing.

## Before you run this on Colab
1. Open directly: `https://colab.research.google.com/github/m00nLeo/judging-the-judges-study/blob/main/notebooks/03_E2_position_bias_protocols.ipynb`
2. Run the next cell (uncomment the clone line) to pull the repo + install deps + re-download the public paper data into `data/upstream/` (gitignored, so every fresh runtime needs this once).
3. Get free keys and add them to **Colab Secrets** (🔑 icon in the left sidebar → "+ Add new secret", toggle "Notebook access" on for each):
   - `GOOGLE_API_KEY` — aistudio.google.com/apikey (free tier, rate-limited, no card needed for testing).
   - `GROQ_API_KEY` — console.groq.com/keys (free, no card).
   Never paste a key into a cell; `src/judges/clients.py` reads these automatically.
4. Run the **dry-run cell** first (zero cost, zero API calls) — must pass before touching a real key.
5. Run the **pilot cell** (20 MT-Bench pairs + all 50 position pairs, one judge at a time via `JUDGE_NAME`) and look at the actual `cost_usd` printed (should be $0.00 on the free judges) before going further.
6. Only then run the **full cell**. The budget cap in `config.yaml` (`budget.e2_position_bias_usd`) still applies if you ever swap in a paid model.
7. When done: send me back `results/e2_position_bias/*.jsonl` (the per-call cache files) — I'll read and analyze from those, not from your printed summary alone.

In [ ]:
# On Colab, uncomment and run this cell BEFORE the cells below.
# !git clone https://github.com/m00nLeo/judging-the-judges-study.git
# %cd judging-the-judges-study

# !pip install -q openai google-genai pyyaml
# !bash scripts/fetch_upstream.sh

In [ ]:
import sys
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
RESULTS_DIR = PROJECT_ROOT / "results" / "e2_position_bias"

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
print(config)

## Step 1 — dry run (zero cost, zero API calls, no key needed)

Validates the whole pipeline (pair loading, caching, budget-cap logic) with a fake judge before any real key touches it.

In [ ]:
from src.experiments.position_bias import (
    load_position_pairs_protocol_a,
    load_mt_bench_sample_protocol_b,
    run_position_bias_protocol,
)
from src.judges.clients import build_judge

position_pairs = load_position_pairs_protocol_a(UPSTREAM / "data" / "custom" / "controlled_pairs.jsonl")
mt_bench_pilot_pairs = load_mt_bench_sample_protocol_b(
    UPSTREAM / "data" / "raw" / "mt_bench" / "mt_bench.jsonl",
    n=config["sampling"]["e2_position_bias"]["mt_bench_pilot_sample"],
    seed=config["seed"],
)
print(f"Protocol A: {len(position_pairs)} position pairs. Protocol B pilot: {len(mt_bench_pilot_pairs)} MT-Bench pairs.")

dry_run_cache = RESULTS_DIR / "_dry_run_cache.jsonl"
dry_run_cache.unlink(missing_ok=True)  # always start the dry run clean
dry_judge = build_judge("dry-run", dry_run=True)
summary = run_position_bias_protocol(
    position_pairs, dry_judge, judge_name="dry-run", protocol="A", cache_path=dry_run_cache, dry_run=True
)
print("Dry-run protocol A summary (fake data, not analyzable):", summary)
assert summary["n_pairs_incomplete"] == 0, "pipeline did not complete all pairs even in dry-run mode"
print("PASS: pipeline runs end to end with zero API calls.")

## Step 2 — pilot run (REAL API calls, small scale)

Runs all 50 Protocol-A pairs + the Protocol-B pilot sample, for ONE judge at a time. Check the printed `cost_usd` sum before moving to Step 3.

In [ ]:
from src.experiments.position_bias import CostModel

JUDGE_NAME = "gemini-3.8-flash"  # was gemini-2.5-flash (retired for new users as of 2026-11); change to "llama-3.3-70b-versatile" for the second judge, run separately

price = config["cost_per_1k_tokens"][JUDGE_NAME]
cost_model = CostModel(cost_per_1k_input=price["input"], cost_per_1k_output=price["output"])
budget = config["budget"]["e2_position_bias_usd"]

judge = build_judge(JUDGE_NAME, dry_run=False)  # reads the key from Colab Secrets / env, never from this cell

cache_a = RESULTS_DIR / f"{JUDGE_NAME}_protocolA_position.jsonl"
cache_b_pilot = RESULTS_DIR / f"{JUDGE_NAME}_protocolB_mtbench_pilot.jsonl"

summary_a = run_position_bias_protocol(
    position_pairs, judge, judge_name=JUDGE_NAME, protocol="A", cache_path=cache_a,
    cost_model=cost_model, budget_usd=budget, dry_run=False,
)
summary_b_pilot = run_position_bias_protocol(
    mt_bench_pilot_pairs, judge, judge_name=JUDGE_NAME, protocol="B", cache_path=cache_b_pilot,
    cost_model=cost_model, budget_usd=budget, dry_run=False,
)

import json
def total_cost(cache_path):
    return sum(json.loads(l)["cost_usd"] for l in cache_path.read_text(encoding="utf-8").splitlines() if l.strip())

print(f"Protocol A ({JUDGE_NAME}): {summary_a}, cost so far: ${total_cost(cache_a):.4f}")
print(f"Protocol B pilot ({JUDGE_NAME}): {summary_b_pilot}, cost so far: ${total_cost(cache_b_pilot):.4f}")
print("\nSTOP HERE. Review the cost above before running the full Protocol B sample in Step 3.")

## Step 3 — full run (only after reviewing Step 2's cost)

Same as Step 2's Protocol B but with `mt_bench_full_sample` pairs. Re-run Step 2's judge selection first for the second judge, then come back here for each judge.

In [ ]:
mt_bench_full_pairs = load_mt_bench_sample_protocol_b(
    UPSTREAM / "data" / "raw" / "mt_bench" / "mt_bench.jsonl",
    n=config["sampling"]["e2_position_bias"]["mt_bench_full_sample"],
    seed=config["seed"],
)
cache_b_full = RESULTS_DIR / f"{JUDGE_NAME}_protocolB_mtbench_full.jsonl"

summary_b_full = run_position_bias_protocol(
    mt_bench_full_pairs, judge, judge_name=JUDGE_NAME, protocol="B", cache_path=cache_b_full,
    cost_model=cost_model, budget_usd=budget, dry_run=False,
)
print(f"Protocol B full ({JUDGE_NAME}): {summary_b_full}, cost so far: ${total_cost(cache_b_full):.4f}")
print(f"\nDone for {JUDGE_NAME}. Repeat Steps 2-3 with JUDGE_NAME = the other judge.")
print("When both judges are done, zip the results/e2_position_bias/ folder (the *.jsonl cache files) and send it back.")